# Food

The food yield system — pure terrain computation, no game pieces.

- `FoodYield` (dataclass) — compute, summary
- `FoodYield.__ft__` — matplotlib tier distribution charts
- `FoodOverlay` — dotted density overlay from food tiers
- `FoodStats` (NamedTuple) — diet, cost, harvest, drain, bonus_sight
- `FOOD_TABLE: dict[PieceType, FoodStats]`
- `LIFESPAN: dict[PieceType, int]`


In [ ]:
#| default_exp magic/food

In [ ]:
#| export
from __future__ import annotations
from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import NamedTuple
from itertools import combinations
import heapq


from fastcore.basics import patch

In [ ]:
#| export
from fasthtml.common import *
from fasthtml.jupyter import *
from fasthtml.common import *

In [ ]:
#| export
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import Terrain, HexGrid, HexPosition
from HexMagic.overlay import CreamOverlay, RiverOverlay,OceanWaveOverlay, HexTouchMap
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns

In [ ]:
showDemo = False

## Constants

In [ ]:
#| export
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int
    cost: int
    harvest: float
    drain: float
    bonus_sight: int
    lifespan: int    # turns before piece expires

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=1, cost=3,  harvest=1.25, drain=1.0,  bonus_sight=0, lifespan=36),
    PieceType.BISHOP: FoodStats(diet=3, cost=9,  harvest=1.0,  drain=0.75, bonus_sight=3, lifespan=5),
    PieceType.KNIGHT: FoodStats(diet=2, cost=10, harvest=0.0,  drain=3.0,  bonus_sight=0, lifespan=5),
    PieceType.QUEEN:  FoodStats(diet=4, cost=18, harvest=0.0,  drain=4.5,  bonus_sight=2, lifespan=20),
    PieceType.KING:   FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1, lifespan=999),
}


## Terrain helpers

In [ ]:
#| export
@patch(as_prop=True)
def land_components(terr: Terrain) -> dict[int, int]:
    """Map each land hex to its connected component ID. Computed once."""
    if hasattr(terr, '_land_comp'):
        return terr._land_comp
    
    comp = {}
    cid = 0
    for i in range(len(terr.elevations)):
        if terr.elevations[i] <= 0 or i in comp:
            continue
        frontier = {i}
        while frontier:
            cur = frontier.pop()
            if cur in comp:
                continue
            comp[cur] = cid
            for nb in terr.hexGrid.neighborsOf(cur):
                if nb not in comp and terr.elevations[nb] > 0:
                    frontier.add(nb)
        cid += 1
    terr._land_comp = comp
    return comp

@patch
def land_connected(self: Terrain, indices: list[int]) -> bool:
    comp = self.land_components
    cids = {comp.get(i) for i in indices}
    return len(cids) == 1 and None not in cids


In [ ]:
#| export
def hexes_in_range(n):
        """ finding all hexes within distance N from origin: """
        results = []
        for q in range(-n, n + 1):
            for r in range(max(-n, -q - n), min(n, -q + n) + 1):
                s = -q - r
                results.append(HexPosition(q, r, s))
        return results

## Food Yield

In [ ]:
#| export


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve:  dict[int, float] = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list[float] = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    


In [ ]:
#| export
@patch
def harvest(self:FoodYield,piece:PieceType,index:int) ->int:
    return int(self.tiers[index] * FOOD_TABLE[piece].harvest)

In [ ]:
#| export
def hexes_in_range(n):
        """ finding all hexes within distance N from origin: """
        results = []
        for q in range(-n, n + 1):
            for r in range(max(-n, -q - n), min(n, -q + n) + 1):
                s = -q - r
                results.append(HexPosition(q, r, s))
        return results

In [ ]:
#| export
@patch
def richness_scores(self: FoodYield, ring_radius: int = 3) -> np.ndarray:
    """Sum of food tiers in the neighborhood of each hex."""
    pattern = hexes_in_range(ring_radius)
    scores, _, _ = self.terrain.convolution_np(
        self.tiers.astype(float), pattern, 1.0, method='sum'
    )
    scores[self.terrain.elevations <= 0] = 0.0
    return scores


In [ ]:
#| export
@patch
def find_oasis(self: FoodYield, n: int, ring_radius: int = 3,
               min_distance: int = 6, balance_tolerance: float = 0.3,
               top_k: int = 40) -> list[int]:
    """Find n well-separated, balanced, high-richness start positions."""
    

    terr = self.terrain
    grid = terr.hexGrid
    scores = self.richness_scores(ring_radius)

    # Top candidates by richness
    ranked = sorted(
        [(i, scores[i]) for i in range(len(scores)) if scores[i] > 0],
        key=lambda x: -x[1]
    )[:top_k]

    def hex_dist(a, b):
        return grid.index_to_hexposition(a).distance(grid.index_to_hexposition(b))

    best_set, best_spread = None, float('inf')

    for combo in combinations(range(len(ranked)), n):
        idxs = [ranked[c][0] for c in combo]
        scs  = [ranked[c][1] for c in combo]

        # Enforce minimum spacing
        if any(hex_dist(a, b) < min_distance for a, b in combinations(idxs, 2)):
            continue

        # All positions must be land-connected
        if not terr.land_connected(idxs):
            continue

        # Balance check
        median = np.median(scs)
        if median == 0: continue
        spread = (max(scs) - min(scs)) / median

        if spread < best_spread:
            best_spread = spread
            best_set = list(zip(idxs, scs))
            if spread <= balance_tolerance:
                break

    if best_set is None:
        raise ValueError(f"Could not find {n} balanced land-connected positions (top_k={top_k})")

    print(f"Oasis positions (spread={best_spread:.2f}):")
    for idx, score in best_set:
        print(f"  Hex {idx}: richness={score:.0f}, tier={self.tiers[idx]}, elev_level={terr.elevationLevel(idx)}")

    return [idx for idx, _ in best_set]


In [ ]:
@patch
def find_oasis(self: FoodYield, n: int, ring_radius: int = 3,
               min_distance: int = 6, balance_tolerance: float = 0.3,
               top_k: int = 40) -> list[int]:
    """Find k largest peaks above min_height with exclusion zones.
    
    Args:
        k: maximum number of peaks to find
        min_height: minimum elevation threshold
        exclusion_radius: how many rings around each peak to exclude
    
    Returns:
        List of indices of peaks (up to k peaks)
    """

    terr = self.terrain
    grid = terr.hexGrid
    scores = self.richness_scores(ring_radius)
 
    # Create max heap of all hexes above threshold
    # Use negative elevation for max heap behavior
    class OSpot(NamedTuple):
        index: int
        score: float
        position: HexPosition


    # Top candidates by richness
    candidates = sorted(
        [OSpot(i, scores[i],grid.index_to_hexposition(index=i)) for i in range(len(scores)) if scores[i] > 0],
        key=lambda x: -x[1]
    )
    ret = []

    
    #print(candidates[:7])
    heapq.heapify(candidates)
    
    peaks = []
    blocked = set()
    itt_n = 0
    
    while len(peaks) < n and candidates and itt_n < len(candidates):
        spot  = heapq.heappop(candidates)
        itt_n += 1
        
        # Skip if this hex is blocked
        if spot.index in blocked:
            continue

        if not terr.land_connected(peaks + [spot.index]):
            continue

        
        # Add this peak
        peaks.append(spot.index)

        # Block this peak and surrounding hexes
        for radius in range(min_distance + 1):
            for blocked_idx in terr.hexGrid.neighborsOf(spot.index, radius):
                blocked.add(blocked_idx)
                
        nPos = [OSpot(x.index, x.score + spot.position.distance(x.position)/1000,x.position) 
                for x in candidates if x.index not in blocked ]
        
        
        candidates = sorted(nPos,key=lambda x: -x[1])

        
        
    
    return peaks

Find oasis heap needs to create an equivalent to 
```python
idxs = [ranked[c][0] for c in combo]
```
for the land_connected part

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're right — `land_connected` expects a **list of indices**, but in `find_oasis_heap` you're passing a single `idx` instead of the accumulated list of peaks so far (plus the current candidate).

The fix would be to pass something like:

```python
if not terr.land_connected(peaks + [idx]):
    continue
```

Does that make sense? Give it a try!

In [ ]:
#| export
@patch
def summary(self:FoodYield,top_n = 5,ring_radius=3):
    """Print tier distribution."""
    land = self.tiers[self.terrain.elevations > 0]
    print(f"Land hexes: {len(land)}")
    for tier in range(self.n_tiers):
        count = np.sum(land == tier)
        bar = '█' * (count // 10)
        print(f"  Tier {tier}: {count:5d} {bar}")

    # Richness overview
    scores = self.richness_scores(ring_radius=ring_radius)
    land_scores = scores[self.terrain.elevations > 0]
    print(f"\nRichness (ring-{ring_radius} neighborhood sum):")
    print(f"  Range:  {land_scores.min():.0f} – {land_scores.max():.0f}")
    print(f"  Mean:   {land_scores.mean():.1f}")
    p25, p50, p75 = np.percentile(land_scores, [25, 50, 75])
    print(f"  p25={p25:.0f}  median={p50:.0f}  p75={p75:.0f}")

    # Quick peek at top candidates
    
    ranked = sorted(
        [(i, scores[i]) for i in range(len(scores)) if scores[i] > 0],
        key=lambda x: -x[1]
    )[:top_n]
    print(f"\n  Top {top_n} oasis candidates:")
    for idx, score in ranked:
        print(f"    Hex {idx}: richness={score:.0f}, tier={self.tiers[idx]}, elev={self.terrain.elevationLevel(idx)}")


In [ ]:
#| export
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
myWorld = TerraDemo().marin_map() #TerraDemo().reykjavik_map()
myWorld.carve_to_ocean(num_lakes=0)
myWorld.compute_climate()

In [ ]:
basins = DrainageBasins(myWorld)

In [ ]:
fy = FoodYield(terrain=myWorld,basins=basins,n_tiers=8)

In [ ]:
fy.compute()

In [ ]:
if showDemo:
    show(fy)
else:
    fy.summary()

In [ ]:
fy.find_oasis(n=3)

In [ ]:
??FoodYield.find_oasis

Is find_oasis_heap working? I think it is finding ocean

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Whoops! An error (timeout) occurred while processing your request.
If this problem persists, please contact us on Discord.
Please include your dialog url and error info in your message.

## Overlay

In [ ]:
#| export
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


Do you think you could write an oasis overlay?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! Before diving in, a couple of quick questions:

1. Should the overlay call `find_oasis` automatically (with configurable `n` and params), or should it accept a pre-computed list of hex indices?
2. How do you want the oasis hexes marked — a solid filled hex, a distinct color, a symbol, or something else?

Let's have it call find oasis based upon a n parameter and a min distance.
I am leaning towards a dotted overlay that would should how good the richness_score is.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great choices! Here's the plan before writing anything:

- `FoodOverlay` uses `ballDensity` patterns keyed by tier — we can do something similar, but instead of per-hex tiers, we'd use the **richness score** (normalized into bands) to pick a dot density for each hex near the oasis positions.

A few decisions to nail down:

1. Should *all* hexes show richness density (like a heatmap), or just those **within a ring** around the chosen oasis points?
2. Should the oasis hex itself be marked distinctly (e.g. a solid or contrasting color dot)?

Also — do you want me to go ahead and write the full overlay, using `FoodOverlay` as a template?

Yes please write using the food overlay as a template, but uses different colors that would work well. in theory these could be shown at the same time

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the overlay, using amber/orange tones to contrast with `FoodOverlay`'s green — they should layer nicely:

```python
def OasisOverlay(color: str = "#E65100", n: int = 3, min_distance: int = 6,
                 ring_radius: int = 3, n_tiers: int = 8,
                 balance_tolerance: float = 0.3, top_k: int = 40, **kw):
    """Dotted richness-score density overlay with oasis positions highlighted."""

    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()

        scores = fy.richness_scores(ring_radius)
        land_mask = ctx.terrain.elevations > 0
        land_scores = scores[land_mask]
        s_min, s_max = land_scores.min(), land_scores.max()

        def score_to_tier(s):
            if s_max == s_min: return 0
            return int((s - s_min) / (s_max - s_min) * (n_tiers - 1))

        oasis_idxs = set(fy.find_oasis(n, ring_radius=ring_radius,
                                        min_distance=min_distance,
                                        balance_tolerance=balance_tolerance,
                                        top_k=top_k))

        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(levels=n_tiers, fills=[color], prefix="oasis_richness")
        oasis_patterns = patGen.ballDensity(levels=1, fills=["#FFD600"], prefix="oasis_spot")

        overlay = ""
        used = set()

        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0: continue
            s = scores[i]
            if s <= 0: continue

            tier = score_to_tier(s)
            used.add(tier)
            pat_name = (oasis_patterns[0].attributes['id'] if i in oasis_idxs
                        else patterns[tier].attributes['id'])

            hex_obj = ctx.grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'

        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        ctx.builder.add_definition(oasis_patterns[0])

        return overlay

    return OverlaySpec("oasis", render, requires={'basins'}, priority=45)
```

A couple of notes:
- Richness is **normalized across all land hexes** into `n_tiers` density bands, so it's a relative heatmap
- Oasis hexes get a bright **yellow** (`#FFD600`) dot to make them pop against both the orange and green layers
- Priority is `45` (just above `FoodOverlay`'s `44`) so it renders on top

Want to try running it before tweaking anything?

In [ ]:
#| export
def OasisOverlay(color: str = "#E65100", n: int = 3, min_distance: int = 6,
                 ring_radius: int = 3, n_tiers: int = 8,
                 balance_tolerance: float = 0.3, top_k: int = 40,showAll=False, **kw):
    """Dotted richness-score density overlay with oasis positions highlighted."""

    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()

        scores = fy.richness_scores(ring_radius)
        land_mask = ctx.terrain.elevations > 0
        land_scores = scores[land_mask]
        s_min, s_max = land_scores.min(), land_scores.max()

        def score_to_tier(s):
            if s_max == s_min: return 0
            return int((s - s_min) / (s_max - s_min) * (n_tiers - 1))

        oasis_idxs = set(fy.find_oasis(n, ring_radius=ring_radius,
                                        min_distance=min_distance,
                                        balance_tolerance=balance_tolerance,
                                        top_k=top_k))

        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(levels=n_tiers, fills=[color], prefix="oasis_richness")
        oasis_patterns = patGen.ballDensity(levels=1, fills=["#FFD600"], prefix="oasis_spot")

        overlay = ""
        used = set()

        if showAll:

            for i in range(len(ctx.terrain.elevations)):
                if ctx.terrain.elevations[i] <= 0: continue
                
                s = scores[i]
                if s <= 0: continue

                tier = score_to_tier(s)
                used.add(tier)
                pat_name = (oasis_patterns[0].attributes['id'] if i in oasis_idxs
                            else patterns[tier].attributes['id'])

                hex_obj = ctx.grid.hexes[i]
                pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
                overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        else:
            for i in oasis_idxs:
                s = scores[i]
                tier = score_to_tier(s)
                used.add(tier)
                pat_name = patterns[tier].attributes['id']
                hex_obj = ctx.grid.hexes[i]
                pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
                overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'

        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        ctx.builder.add_definition(oasis_patterns[0])

        return overlay

    return OverlaySpec("oasis", render, requires={'basins'}, priority=45)


In [ ]:
oasis_idxs = set(fy.find_oasis(3,min_distance=10))

In [ ]:
oasis_idxs

In [ ]:
#| export
def HexNumberOverlay(font_size: int = 0, color: str = "#333", opacity: float = 0.5, **kw) -> OverlaySpec:
    """Label each hex with its grid index, using coarse index when zoomed."""
    def render(ctx) -> str:
        grid = ctx.grid
        r = grid.radius
        fs = font_size or max(6, int(r * 0.35))

        # Build fine→coarse reverse map if zoomed
        f2c = None
        if ctx.c2f is not None:
            f2c = {fi: ci for ci, fis in ctx.c2f.items() for fi in fis}

        parts = []
        for i, h in enumerate(grid.hexes):
            label = f2c.get(i, i) if f2c is not None else i
            c = h.center
            parts.append(
                f'<text x="{c.x:.1f}" y="{c.y:.1f}" '
                f'text-anchor="middle" dominant-baseline="central" '
                f'font-size="{fs}" fill="{color}" opacity="{opacity}" '
                f'font-family="monospace">{label}</text>'
            )

        return '\n'.join(parts)

    return OverlaySpec("hex_numbers", render, priority=95)


So HexNumberOverlay should have the hexnumber from the parent and not from the ctxgrid. Does this make sense?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, that makes perfect sense! The oasis indices (like `{1200, 9, 4065}`) come from the **parent/coarse** `FoodYield` computation, so if you're viewing a zoomed grid the labels need to match those coarse indices — otherwise you can't visually verify which hexes were selected.

The code already has a stub for this:
```python
coarse = ctx.coarse_index(i) if hasattr(ctx, 'coarse_index') else i
```

Do you have a `coarse_index` method on `ctx`, or does the mapping live somewhere else (e.g. on the grid itself)?

Is there anything from the overlay file that could tell about reversing the map

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! Looking at `OverlayContext`, there's a `c2f` dict (coarse-to-fine) and two helper methods:

- `fine_indices(coarse_idx)` → list of fine indices
- `fine_first(coarse_idx)` → first fine index, or -1

There's **no built-in fine→coarse reverse map**, but since `c2f` is structured as `{old_idx: [new_idx]}`, you could invert it yourself with something like:

```python
f2c = {fi: ci for ci, fis in ctx.c2f.items() for fi in fis}
```

Is that the direction you're thinking — looking up the coarse index from a fine one (e.g. to label a zoomed hex with its parent index)?

Yes that is the direction.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

So in `HexNumberOverlay`, you'd build that reverse map and use it to label each fine hex with its coarse index:

```python
f2c = {fi: ci for ci, fis in ctx.c2f.items() for fi in fis}
coarse = f2c.get(i, i)
```

Want to try plugging that into your `HexNumberOverlay` yourself?

So what I am wondering is about HexTouchMap for an inner map. we want it when we touch the overlay we return the touch position to the map,

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The key insight is that `on_click(grid, i)` gets the **fine index**, but you want to report the **coarse index** back to the parent map.

The cleanest approach would be a wrapper that intercepts the fine index, looks it up in `f2c`, and passes the coarse index to the real callback:

```python
def coarse_click(ctx, on_click):
    """Wrap an on_click so it receives the coarse (parent) index instead of fine."""
    if ctx.c2f is None:
        return on_click  # not zoomed, pass through unchanged
    f2c = {fi: ci for ci, fis in ctx.c2f.items() for fi in fis}
    def wrapped(grid, fine_idx):
        coarse_idx = f2c.get(fine_idx, fine_idx)
        return on_click(grid, coarse_idx)
    return wrapped
```

Then in `TerrainDisplay` or wherever you call `HexTouchMap`, you'd use:

```python
HexTouchMap(ctx.grid, on_click=coarse_click(ctx, my_handler), ...)
```

That way the inner zoomed map fires clicks with coordinates meaningful to the **parent terrain** — the same indices that `find_oasis` etc. work with. Does that match what you had in mind?

In [ ]:
ctx = GameContext(
        terrain=myWorld, grid=myWorld.hexGrid, builder=myWorld.hexGrid.builder,
        c2f=getattr(myWorld, 'c2f', None),
        extras=dict(
            basins=basins,
            coarse_basins=basins
        )
    )

In [ ]:
def oasis_heap(self: FoodYield, n: int, ring_radius: int = 3,
               min_distance: int = 6, balance_tolerance: float = 0.3,
               top_k: int = 40) -> list[int]:
    """Find k largest peaks above min_height with exclusion zones.
    
    Args:
        k: maximum number of peaks to find
        min_height: minimum elevation threshold
        exclusion_radius: how many rings around each peak to exclude
    
    Returns:
        List of indices of peaks (up to k peaks)
    """
 
    # Create max heap of all hexes above threshold
    # Use negative elevation for max heap behavior
    class OSpot(NamedTuple):
    """Per-piece-type food economics."""
        index: int
        score: float
        position: HexPosition

    

    # Top candidates by richness
    candidates = sorted(
        [OSpot(i, scores[i],grid.index_to_hexposition(index=i) for i in range(len(scores)) if scores[i] > 0],
        key=lambda x: -x[1]
    )
    ret = []

    
    print(candidates[:7])
    heapq.heapify(candidates)
    
    peaks = []
    blocked = set()
    itt_n = 0
    
    while len(peaks) < n and candidates and itt_n < len(candidates):
        spot  = heapq.heappop(candidates)
        itt_n += 1
        
        # Skip if this hex is blocked
        if spot.index in blocked:
            continue

        if not terr.land_connected(peaks + [spot.index]):
            continue

        
        # Add this peak
        peaks.append(spot.index)

        # Block this peak and surrounding hexes
        for radius in range(min_distance + 1):
            for blocked_idx in terr.hexGrid.neighborsOf(spot.index, radius):
                blocked.add(blocked_idx)
                
        nPos = [OSpot(x.index, x.score + spot.distance(x.position)/100,x.position) 
                for x in candidates if x.index not in blocked ]
        
        
        candidates = sorted(nPos,key=lambda x: -x[1])

        
        
    
    return peaks

In [ ]:
myWorld.hexGrid.builder.layers = []
TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        #FoodOverlay(),
        HexNumberOverlay(),
        OasisOverlay(n=8),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        #debug = not showDemo
    )

can you write this

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

There are a few bugs to fix — missing `self` setup, a misplaced docstring, and a bracket mismatch. Here's the corrected version:

```python
@patch
def find_oasis(self: FoodYield, n: int, ring_radius: int = 3,
               min_distance: int = 6, balance_tolerance: float = 0.3,
               top_k: int = 40) -> list[int]:
    """Find n well-separated high-richness start positions using a greedy heap."""

    terr = self.terrain
    grid = terr.hexGrid
    scores = self.richness_scores(ring_radius)

    class OSpot(NamedTuple):
        index: int
        score: float
        position: HexPosition

    candidates = sorted(
        [OSpot(i, scores[i], grid.index_to_hexposition(index=i))
         for i in range(len(scores)) if scores[i] > 0],
        key=lambda x: -x[1]
    )

    heapq.heapify(candidates)

    peaks = []
    blocked = set()
    itt_n = 0

    while len(peaks) < n and candidates and itt_n < len(candidates):
        spot = heapq.heappop(candidates)
        itt_n += 1

        if spot.index in blocked:
            continue

        if not terr.land_connected(peaks + [spot.index]):
            continue

        peaks.append(spot.index)

        for radius in range(min_distance + 1):
            for blocked_idx in terr.hexGrid.neighborsOf(spot.index, radius):
                blocked.add(blocked_idx)

        candidates = sorted(
            [OSpot(x.index, x.score + spot.position.distance(x.position) / 100, x.position)
             for x in candidates if x.index not in blocked],
            key=lambda x: -x[1]
        )

    return peaks
```

Key fixes:
- Added `terr`, `grid`, `scores` from `self`
- Removed misplaced docstring inside `NamedTuple`
- Fixed the bracket mismatch in the list comprehension
- Changed `spot.distance(...)` → `spot.position.distance(...)`